# AdaBoost Malware Classification with Balanced Error Rate

This notebook trains and evaluates AdaBoost classifiers on the reduced Android malware data set (10,000 samples) to design a model that minimizes balanced error rate (BER).

In [1]:
import json
import itertools
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

warnings.simplefilter("ignore", category=FutureWarning)
RANDOM_SEED = 42


In [2]:
# Load the data
file_path = "AndroidmalwareSmall.xlsx"
df = pd.read_excel(file_path)

X = df.drop(columns=["Result"])
y = df["Result"].astype(int)

print(f"Dataset shape: {df.shape}")
print("Number of features (excluding target):", df.shape[1] - 1)
print("Missing values in the data set:", int(df.isna().sum().sum()))

class_distribution = df["Result"].value_counts().sort_index()
print("\nClass distribution (counts):")
print(class_distribution)
print("\nClass distribution (proportions):")
print((class_distribution / class_distribution.sum()).round(4))


Dataset shape: (10000, 87)
Number of features (excluding target): 86
Missing values in the data set: 0

Class distribution (counts):
Result
0    5044
1    4956

Class distribution (proportions):
Result
0    0.5044
1    0.4956


## Baseline AdaBoost model
We begin with the scikit-learn defaults to understand baseline balanced accuracy and BER.

In [3]:
baseline_clf = AdaBoostClassifier(random_state=RANDOM_SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
baseline_scores = []
for train_idx, test_idx in cv.split(X, y):
    baseline_clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = baseline_clf.predict(X.iloc[test_idx])
    baseline_scores.append(balanced_accuracy_score(y.iloc[test_idx], preds))

baseline_scores = np.array(baseline_scores)
baseline_bal_acc = baseline_scores.mean()
baseline_ber = 1.0 - baseline_bal_acc

print("Baseline balanced accuracy scores (per fold):", np.round(baseline_scores, 4))
print(f"Baseline mean balanced accuracy: {baseline_bal_acc:.4f}")
print(f"Baseline balanced error rate: {baseline_ber:.4f}")


Baseline balanced accuracy scores (per fold): [0.9536 0.9487 0.9571 0.9417 0.9376]
Baseline mean balanced accuracy: 0.9477
Baseline balanced error rate: 0.0523


## Hyper-parameter tuning with nested cross-validation
A manual 3x2 nested stratified cross-validation (three outer folds, two inner folds) tunes AdaBoost hyper-parameters while focusing on balanced accuracy for this 10k-sample data set.

In [4]:
from itertools import product

param_grid = list(product([100, 200], [0.5, 1.0], [1, 2]))
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_SEED)

outer_records = []
for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    best_score = -np.inf
    best_params = None
    for n_estimators, learning_rate, max_depth in param_grid:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train):
            X_inner_train, X_inner_val = X_train.iloc[inner_train_idx], X_train.iloc[inner_val_idx]
            y_inner_train, y_inner_val = y_train.iloc[inner_train_idx], y_train.iloc[inner_val_idx]

            tree = DecisionTreeClassifier(max_depth=max_depth, random_state=RANDOM_SEED, min_samples_leaf=1)
            model = AdaBoostClassifier(
                estimator=tree,
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                algorithm="SAMME",
                random_state=RANDOM_SEED,
            )
            model.fit(X_inner_train, y_inner_train)
            inner_pred = model.predict(X_inner_val)
            inner_scores.append(balanced_accuracy_score(y_inner_val, inner_pred))

        inner_mean = float(np.mean(inner_scores))
        if inner_mean > best_score:
            best_score = inner_mean
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "algorithm": "SAMME",
                "estimator__max_depth": max_depth,
                "estimator__min_samples_leaf": 1,
            }

    final_tree = DecisionTreeClassifier(
        max_depth=best_params["estimator__max_depth"],
        min_samples_leaf=1,
        random_state=RANDOM_SEED,
    )
    tuned_model = AdaBoostClassifier(
        estimator=final_tree,
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["learning_rate"],
        algorithm=best_params["algorithm"],
        random_state=RANDOM_SEED,
    )
    tuned_model.fit(X_train, y_train)
    fold_pred = tuned_model.predict(X_test)
    outer_bal_acc = balanced_accuracy_score(y_test, fold_pred)

    outer_records.append({
        "fold": fold_idx,
        "balanced_accuracy": outer_bal_acc,
        "balanced_error_rate": 1.0 - outer_bal_acc,
        "best_params": best_params,
    })

outer_df = pd.DataFrame(outer_records)
outer_df["params_key"] = outer_df["best_params"].apply(lambda params: tuple(sorted(params.items())))
params_stats = (
    outer_df.groupby("params_key")
    .agg(mean_balanced_accuracy=("balanced_accuracy", "mean"), folds=("balanced_accuracy", "count"))
    .sort_values(by=["mean_balanced_accuracy", "folds"], ascending=[False, False])
)

mean_bal_acc = outer_df["balanced_accuracy"].mean()
std_bal_acc = outer_df["balanced_accuracy"].std(ddof=1)
mean_ber = outer_df["balanced_error_rate"].mean()
std_ber = outer_df["balanced_error_rate"].std(ddof=1)

print("Per-fold performance:")
print(outer_df[["fold", "balanced_accuracy", "balanced_error_rate"]])
print(f"\nMean balanced accuracy: {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")
print(f"Mean balanced error rate: {mean_ber:.4f} ± {std_ber:.4f}")

params_summary = params_stats.reset_index()
params_summary["params"] = params_summary["params_key"].apply(lambda key: dict(key))
print("\nParameter combinations observed during nested CV:")
print(params_summary[["mean_balanced_accuracy", "folds", "params"]])

recommended_params = dict(params_stats.index[0])


Per-fold performance:
 fold  balanced_accuracy  balanced_error_rate
    1           0.961367             0.038633
    2           0.961071             0.038929
    3           0.948485             0.051515

Mean balanced accuracy: 0.9570 ± 0.0074
Mean balanced error rate: 0.0430 ± 0.0074

Parameter combinations observed during nested CV:
 mean_balanced_accuracy  folds                                                                                                                         params
               0.956974      3 {'algorithm': 'SAMME', 'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'learning_rate': 1.0, 'n_estimators': 200}


In [5]:
ada_params = {k: v for k, v in recommended_params.items() if not k.startswith("estimator__")}
estimator_params = {k.split("__", 1)[1]: v for k, v in recommended_params.items() if k.startswith("estimator__")}

final_tree = DecisionTreeClassifier(random_state=RANDOM_SEED, **estimator_params)
final_model = AdaBoostClassifier(estimator=final_tree, random_state=RANDOM_SEED, **ada_params)
final_model.fit(X, y)

print("Recommended hyperparameters:")
print(json.dumps(recommended_params, indent=2))
print("\nFinal model parameters (including defaults not changed):")
print(json.dumps({k: final_model.get_params()[k] for k in ["algorithm", "learning_rate", "n_estimators", "random_state"]}, indent=2))
print(json.dumps({k: final_model.get_params()["estimator"].get_params()[k] for k in ["max_depth", "min_samples_leaf", "random_state"]}, indent=2))

print(f"\nEstimated generalization balanced error rate: {mean_ber:.4f} ± {std_ber:.4f}")
print(f"Corresponding balanced accuracy estimate: {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")


Recommended hyperparameters:
{
  "algorithm": "SAMME",
  "estimator__max_depth": 2,
  "estimator__min_samples_leaf": 1,
  "learning_rate": 1.0,
  "n_estimators": 200
}

Final model parameters (including defaults not changed):
{
  "algorithm": "SAMME",
  "learning_rate": 1.0,
  "n_estimators": 200,
  "random_state": 42
}
{
  "max_depth": 2,
  "min_samples_leaf": 1,
  "random_state": 42
}

Estimated generalization balanced error rate: 0.0430 ± 0.0074
Corresponding balanced accuracy estimate: 0.9570 ± 0.0074


## Summary of model exploration and validation
- **Models explored:**
  - Baseline AdaBoostClassifier with default parameters.
  - Nested cross-validated grid search implemented manually (three outer folds, two inner folds) over learning rate, number of estimators, boosting algorithm, and decision tree depth.
- **Generalization error estimation:** 3x2 nested stratified cross-validation optimized balanced accuracy; reported balanced error rate is the complement of the outer-fold balanced accuracy.
- **Parameter deviations from defaults:** Listed above in the recommended hyperparameters; key changes include `n_estimators`, `learning_rate`, boosting `algorithm`, and the decision tree's `max_depth`.